# Aula 2 - Mastering Machine Learning Advanced

## Cross-Validation & Hyperparameter Tuning

### Prof. Dr. Ahirton Lopes (profahirton.lopes@fiap.com.br)

Ref. IBM Telco Customer Churn — https://community.ibm.com/community/user/businessanalytics/blogs/steven-macko/2019/07/11/telco-customer-churn-1113

## Índice

1. [Introdução](#1)
2. [O Problema do Train/Test Split Simples](#2)
3. [K-Fold Cross-Validation](#3)
4. [Stratified K-Fold](#4)
5. [Métricas Além da Acurácia](#5)
   - 5.1 [Precision, Recall e F1-Score](#51)
   - 5.2 [AUC-ROC](#52)
6. [Learning Curves](#6)
7. [Hyperparameter Tuning](#7)
   - 7.1 [Grid Search CV](#71)
   - 7.2 [Randomized Search CV](#72)
   - 7.3 [Optuna](#73)
8. [Comparativo Final de Modelos](#8)
9. [Conclusão](#9)

# 1. Introdução <a id="1"></a>

Na aula anterior preparamos nossos dados com Feature Engineering. Agora vem a pergunta que todo projeto de ML precisa responder:

> **"Esse modelo realmente aprendeu, ou só memorizou os dados de treino?"**

Avaliar um modelo corretamente é tão importante quanto construí-lo. Um modelo que parece ótimo no treino e falha na produção é pior do que não ter modelo nenhum.

Nesta aula vamos cobrir:

- **Cross-Validation** — como estimar a performance real do modelo de forma confiável
- **Métricas além da acurácia** — por que acurácia sozinha engana, especialmente em churn
- **Learning Curves** — diagnosticar overfitting e underfitting
- **Grid Search, Randomized Search e Optuna** — encontrar os melhores hiperparâmetros de forma sistemática

**Contexto de negócio:** continuamos com o dataset Telecom Churn. Identificar o melhor modelo para prever cancelamento é crítico: cada ponto percentual de recall a mais significa clientes retidos que poderiam ter sido perdidos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold,
    cross_val_score, cross_validate,
    GridSearchCV, RandomizedSearchCV,
    learning_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from scipy.stats import randint, uniform

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

np.random.seed(42)

In [ ]:
# carregando e preparando o dataset (mesmo pipeline da Aula 1)
url = 'https://raw.githubusercontent.com/ahirtonlopes/Mastering-Machine-Learning/main/Bases/TelcoChurn.csv'
df = pd.read_csv(url)

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
df = df.drop(columns=['customerID'])

cols_num = ['tenure', 'MonthlyCharges', 'TotalCharges']
cols_cat = ['gender', 'Partner', 'Dependents', 'PhoneService',
            'MultipleLines', 'InternetService', 'OnlineSecurity',
            'OnlineBackup', 'DeviceProtection', 'TechSupport',
            'StreamingTV', 'StreamingMovies', 'Contract',
            'PaperlessBilling', 'PaymentMethod']

X = df[cols_num + cols_cat]
y = df['Churn']

print(f'Shape: {X.shape}')
print(f'Taxa de Churn: {y.mean()*100:.1f}%')

In [ ]:
# preprocessador padrão — reutilizamos o mesmo da Aula 1
pipeline_num = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

pipeline_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', pipeline_num, cols_num),
    ('cat', pipeline_cat, cols_cat)
])

# 2. O Problema do Train/Test Split Simples <a id="2"></a>

O split 80/20 que usamos na Aula 1 tem um problema sério: o resultado depende de **qual 20% foi parar no teste**. Dependendo do split, o mesmo modelo pode parecer ótimo ou ruim.

In [ ]:
# demonstrando a variação de acurácia com diferentes seeds
modelo = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

acuracias = []
for seed in range(20):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
    modelo.fit(X_train, y_train)
    acuracias.append(accuracy_score(y_test, modelo.predict(X_test)))

plt.figure(figsize=(10, 4))
plt.plot(acuracias, marker='o', color='steelblue', linewidth=1.5)
plt.axhline(np.mean(acuracias), color='coral', linestyle='--', label=f'Média: {np.mean(acuracias):.4f}')
plt.fill_between(range(20),
                 np.mean(acuracias) - np.std(acuracias),
                 np.mean(acuracias) + np.std(acuracias),
                 alpha=0.2, color='coral')
plt.title('Acurácia com 20 splits diferentes — mesmos dados, mesmo modelo')
plt.xlabel('Seed do split')
plt.ylabel('Acurácia')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Mínimo: {min(acuracias):.4f} | Máximo: {max(acuracias):.4f} | Desvio padrão: {np.std(acuracias):.4f}')

A variação entre os splits mostra que um único split não é confiável para estimar a performance real. Precisamos de uma estratégia mais robusta.

# 3. K-Fold Cross-Validation <a id="3"></a>

A ideia do K-Fold é dividir o dataset em K partes (folds). A cada rodada, um fold diferente é usado como teste e os K-1 restantes como treino. O resultado final é a média das K avaliações.

![K-Fold](https://i.imgur.com/Wol2oJt.png)

In [ ]:
# K-Fold com 5 folds
kf = KFold(n_splits=5, shuffle=True, random_state=42)

modelo_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

scores_kfold = cross_val_score(modelo_lr, X, y, cv=kf, scoring='accuracy')

print('Acurácia por fold:', np.round(scores_kfold, 4))
print(f'Média: {scores_kfold.mean():.4f} ± {scores_kfold.std():.4f}')

In [ ]:
# visualizando os scores por fold
plt.figure(figsize=(8, 4))
plt.bar(range(1, 6), scores_kfold, color='steelblue', edgecolor='white')
plt.axhline(scores_kfold.mean(), color='coral', linestyle='--',
            label=f'Média: {scores_kfold.mean():.4f}')
plt.title('Acurácia por Fold — K-Fold (K=5)')
plt.xlabel('Fold')
plt.ylabel('Acurácia')
plt.ylim(0.75, 0.85)
plt.legend()
plt.tight_layout()
plt.show()

# 4. Stratified K-Fold <a id="4"></a>

No dataset de Churn, apenas ~26% dos clientes cancelaram. Se um fold cair com poucos casos de churn, a avaliação fica distorcida.

O **Stratified K-Fold** garante que a proporção de classes seja preservada em cada fold — essencial para datasets desbalanceados.

In [ ]:
# comparando distribuição de churn nos folds: KFold vs StratifiedKFold
kf_normal = KFold(n_splits=5, shuffle=True, random_state=42)
kf_strat  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X_arr = X.values
y_arr = y.values

print('KFold — % de Churn por fold:')
for i, (_, test_idx) in enumerate(kf_normal.split(X_arr, y_arr)):
    pct = y_arr[test_idx].mean() * 100
    print(f'  Fold {i+1}: {pct:.1f}%')

print('\nStratifiedKFold — % de Churn por fold:')
for i, (_, test_idx) in enumerate(kf_strat.split(X_arr, y_arr)):
    pct = y_arr[test_idx].mean() * 100
    print(f'  Fold {i+1}: {pct:.1f}%')

In [ ]:
# usando Stratified K-Fold para avaliação mais justa
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores_skf = cross_val_score(modelo_lr, X, y, cv=skf, scoring='accuracy')

print('Acurácia por fold (Stratified):', np.round(scores_skf, 4))
print(f'Média: {scores_skf.mean():.4f} ± {scores_skf.std():.4f}')

# 5. Métricas Além da Acurácia <a id="5"></a>

Se 74% dos clientes não cancelam, um modelo que prevê **sempre "Não Churn"** teria 74% de acurácia — sem aprender nada. Para churn, o que importa é capturar os clientes que vão sair.

## 5.1 Precision, Recall e F1-Score <a id="51"></a>

- **Precision**: dos clientes que o modelo apontou como churn, quantos realmente saíram?
- **Recall**: dos clientes que realmente saíram, quantos o modelo identificou?
- **F1-Score**: média harmônica entre Precision e Recall

> Em churn, **Recall é a métrica mais crítica** — custa mais não identificar um cliente que vai sair do que acionar um que ficaria.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

modelo_lr.fit(X_train, y_train)
y_pred = modelo_lr.predict(X_test)

print(classification_report(y_test, y_pred, target_names=['Não Churn', 'Churn']))

In [ ]:
# matriz de confusão
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Não Churn', 'Churn'],
    cmap='Blues', ax=ax
)
plt.title('Matriz de Confusão — Regressão Logística')
plt.tight_layout()
plt.show()

## 5.2 AUC-ROC <a id="52"></a>

A curva ROC mostra o trade-off entre a taxa de verdadeiros positivos (Recall) e a taxa de falsos positivos para diferentes limiares de classificação. A área sob a curva (AUC) resume isso em um único número — quanto mais próximo de 1, melhor.

In [ ]:
# calculando curva ROC
y_prob = modelo_lr.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='steelblue', lw=2, label=f'Regressão Logística (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Baseline aleatório')
plt.xlabel('Taxa de Falsos Positivos')
plt.ylabel('Taxa de Verdadeiros Positivos (Recall)')
plt.title('Curva ROC — Telecom Churn')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# avaliando múltiplas métricas com cross_validate
metricas = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

resultado_cv = cross_validate(modelo_lr, X, y, cv=skf, scoring=metricas)

print('Resultados com Stratified K-Fold (5 folds) — Regressão Logística:\n')
for metrica in metricas:
    valores = resultado_cv[f'test_{metrica}']
    print(f'  {metrica:<12}: {valores.mean():.4f} ± {valores.std():.4f}')

# 6. Learning Curves <a id="6"></a>

A Learning Curve mostra como a performance do modelo evolui conforme aumentamos o volume de dados de treino. É a ferramenta principal para diagnosticar **overfitting** (variância alta) e **underfitting** (viés alto).

- **Overfitting**: score de treino alto, score de validação baixo — gap grande entre as curvas
- **Underfitting**: ambas as curvas baixas — o modelo não tem capacidade suficiente
- **Ideal**: as duas curvas convergem para um valor alto

In [ ]:
def plot_learning_curve(estimator, X, y, titulo, cv=5, scoring='roc_auc'):
    tamanhos, scores_treino, scores_val = learning_curve(
        estimator, X, y,
        cv=StratifiedKFold(n_splits=cv, shuffle=True, random_state=42),
        scoring=scoring,
        train_sizes=np.linspace(0.1, 1.0, 8),
        n_jobs=-1
    )

    media_treino = scores_treino.mean(axis=1)
    std_treino   = scores_treino.std(axis=1)
    media_val    = scores_val.mean(axis=1)
    std_val      = scores_val.std(axis=1)

    plt.figure(figsize=(9, 5))
    plt.plot(tamanhos, media_treino, 'o-', color='steelblue', label='Treino')
    plt.fill_between(tamanhos, media_treino - std_treino, media_treino + std_treino, alpha=0.15, color='steelblue')
    plt.plot(tamanhos, media_val, 'o-', color='coral', label='Validação')
    plt.fill_between(tamanhos, media_val - std_val, media_val + std_val, alpha=0.15, color='coral')
    plt.title(f'Learning Curve — {titulo}')
    plt.xlabel('Tamanho do conjunto de treino')
    plt.ylabel(scoring)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Regressão Logística
plot_learning_curve(modelo_lr, X, y, 'Regressão Logística')

In [ ]:
# Árvore de Decisão sem restrição — exemplo clássico de overfitting
modelo_dt = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(random_state=42))
])

plot_learning_curve(modelo_dt, X, y, 'Árvore de Decisão (sem restrição)')

O gap grande entre treino e validação na Árvore de Decisão é o sinal clássico de overfitting. Isso nos mostra que precisamos regularizar o modelo — é exatamente o que o Hyperparameter Tuning vai fazer.

# 7. Hyperparameter Tuning <a id="7"></a>

Hiperparâmetros são as configurações do modelo que definimos antes do treino — a profundidade de uma árvore, o número de estimadores de uma floresta, o parâmetro C de uma regressão logística. Encontrar a combinação certa muda completamente o resultado.

## 7.1 Grid Search CV <a id="71"></a>

Testa **todas as combinações** de hiperparâmetros definidas em uma grade. Garante encontrar o ótimo dentro do espaço definido, mas pode ser lento para espaços grandes.

In [ ]:
# Grid Search para Random Forest
modelo_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [5, 10, None],
    'model__min_samples_leaf': [1, 5]
}

grid_search = GridSearchCV(
    modelo_rf,
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

In [ ]:
print('Melhores hiperparâmetros (Grid Search):')
print(grid_search.best_params_)
print(f'\nMelhor AUC-ROC (validação cruzada): {grid_search.best_score_:.4f}')

# performance no teste
y_pred_gs = grid_search.predict(X_test)
y_prob_gs = grid_search.predict_proba(X_test)[:, 1]
print(f'AUC-ROC no teste: {roc_auc_score(y_test, y_prob_gs):.4f}')

In [ ]:
# visualizando os resultados do Grid Search
resultados_gs = pd.DataFrame(grid_search.cv_results_)

pivot = resultados_gs.pivot_table(
    values='mean_test_score',
    index='param_model__max_depth',
    columns='param_model__n_estimators'
)

plt.figure(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd')
plt.title('Grid Search — AUC-ROC por max_depth e n_estimators')
plt.tight_layout()
plt.show()

## 7.2 Randomized Search CV <a id="72"></a>

Em vez de testar todas as combinações, amostra aleatoriamente um número fixo de configurações. Muito mais rápido para espaços grandes e, na prática, encontra resultados comparáveis ao Grid Search com uma fração do tempo.

In [ ]:
# Randomized Search com um espaço de busca maior
param_dist = {
    'model__n_estimators':    randint(50, 400),
    'model__max_depth':       [3, 5, 10, 15, 20, None],
    'model__min_samples_leaf': randint(1, 20),
    'model__max_features':    ['sqrt', 'log2', 0.5]
}

random_search = RandomizedSearchCV(
    modelo_rf,
    param_dist,
    n_iter=30,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X_train, y_train)

In [ ]:
print('Melhores hiperparâmetros (Randomized Search):')
print(random_search.best_params_)
print(f'\nMelhor AUC-ROC (validação cruzada): {random_search.best_score_:.4f}')

y_prob_rs = random_search.predict_proba(X_test)[:, 1]
print(f'AUC-ROC no teste: {roc_auc_score(y_test, y_prob_rs):.4f}')

## 7.3 Optuna <a id="73"></a>

O **Optuna** é o estado da arte em otimização de hiperparâmetros. Usa algoritmos Bayesianos para guiar a busca — aprende com cada tentativa e foca nas regiões mais promissoras do espaço, sendo muito mais eficiente que Grid Search e Randomized Search.

In [ ]:
!pip install optuna -q

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# preparando dados já pré-processados para o Optuna
preprocessor_fit = preprocessor.__class__(
    transformers=preprocessor.transformers
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 50, 400),
        'max_depth':        trial.suggest_int('max_depth', 3, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features':     trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'random_state': 42
    }

    rf = RandomForestClassifier(**params)
    scores = cross_val_score(
        rf, X_train_proc, y_train,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='roc_auc'
    )
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=40, show_progress_bar=True)

In [ ]:
print('Melhores hiperparâmetros (Optuna):')
print(study.best_params)
print(f'\nMelhor AUC-ROC (validação cruzada): {study.best_value:.4f}')

# treinando o modelo final com os melhores parâmetros
rf_optuna = RandomForestClassifier(**study.best_params, random_state=42)
rf_optuna.fit(X_train_proc, y_train)

y_prob_optuna = rf_optuna.predict_proba(X_test_proc)[:, 1]
print(f'AUC-ROC no teste: {roc_auc_score(y_test, y_prob_optuna):.4f}')

In [ ]:
# histórico de otimização do Optuna
trials_df = study.trials_dataframe()

plt.figure(figsize=(10, 4))
plt.plot(trials_df['number'], trials_df['value'], alpha=0.5, color='steelblue', marker='o', markersize=4)
plt.plot(trials_df['number'],
         trials_df['value'].cummax(),
         color='coral', linewidth=2, label='Melhor até agora')
plt.title('Optuna — Evolução da Otimização')
plt.xlabel('Trial')
plt.ylabel('AUC-ROC')
plt.legend()
plt.tight_layout()
plt.show()

# 8. Comparativo Final de Modelos <a id="8"></a>

Agora comparamos todos os modelos avaliados, com as configurações otimizadas.

In [ ]:
# modelos para comparação
modelos_comparar = {
    'Regressão Logística': Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(max_iter=1000))
    ]),
    'Árvore de Decisão (sem tuning)': Pipeline([
        ('preprocessor', preprocessor),
        ('model', DecisionTreeClassifier(random_state=42))
    ]),
    'Random Forest (Grid Search)': grid_search.best_estimator_,
    'Random Forest (Randomized)':  random_search.best_estimator_,
}

skf_eval = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultados = []

for nome, modelo in modelos_comparar.items():
    cv_res = cross_validate(modelo, X, y, cv=skf_eval,
                            scoring=['accuracy', 'recall', 'f1', 'roc_auc'])
    resultados.append({
        'Modelo': nome,
        'Acurácia':  cv_res['test_accuracy'].mean(),
        'Recall':    cv_res['test_recall'].mean(),
        'F1-Score':  cv_res['test_f1'].mean(),
        'AUC-ROC':   cv_res['test_roc_auc'].mean()
    })

df_resultados = pd.DataFrame(resultados).set_index('Modelo').round(4)
df_resultados.sort_values('AUC-ROC', ascending=False)

In [ ]:
# visualizando o comparativo
df_resultados.plot(
    kind='bar', figsize=(12, 5),
    colormap='Set2', edgecolor='white'
)
plt.title('Comparativo de Modelos — Telecom Churn (Stratified K-Fold 5x)')
plt.ylabel('Score')
plt.xticks(rotation=15, ha='right')
plt.ylim(0.5, 1.0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# curvas ROC de todos os modelos no conjunto de teste
plt.figure(figsize=(8, 6))

cores = ['steelblue', 'coral', 'seagreen', 'mediumpurple']

for (nome, modelo), cor in zip(modelos_comparar.items(), cores):
    modelo.fit(X_train, y_train)
    y_prob = modelo.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, color=cor, lw=2, label=f'{nome} (AUC={auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Baseline')
plt.xlabel('Taxa de Falsos Positivos')
plt.ylabel('Recall (Taxa de Verdadeiros Positivos)')
plt.title('Curvas ROC — Comparativo de Modelos')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# 9. Conclusão <a id="9"></a>

Nesta aula cobrimos o ciclo completo de avaliação e otimização de modelos:

| Técnica | O que resolve |
|---|---|
| Train/test split simples | Rápido, mas instável — depende do split |
| K-Fold Cross-Validation | Estimativa mais confiável da performance real |
| Stratified K-Fold | Garante proporção de classes em datasets desbalanceados |
| Precision, Recall, F1 | Avaliação além da acurácia — essencial para churn |
| AUC-ROC | Métrica robusta ao limiar de decisão |
| Learning Curves | Diagnóstico de overfitting e underfitting |
| Grid Search CV | Busca exaustiva no espaço de hiperparâmetros |
| Randomized Search CV | Busca amostrada — mais rápida para espaços grandes |
| Optuna | Otimização Bayesiana — eficiente e estado da arte |

**Regra de ouro:** sempre use Stratified K-Fold para problemas de classificação com classes desbalanceadas. E nunca escolha o modelo pela acurácia — escolha pelo Recall ou AUC-ROC, dependendo do custo do erro para o negócio.

Na próxima aula vamos explorar **Séries Temporais** — como prever o consumo de dados e a receita de clientes ao longo do tempo.

---
**Prof. Dr. Ahirton Lopes** | [LinkedIn](https://linkedin.com/in/ahirtonlopes) | [GitHub](https://github.com/ahirtonlopes)